# Reproducing the results of *Formal Security of Proactive Redistribution for Secret Sharing with Essential Participants under Threshold Changes*

Sudhir Singh and Upasana Singh.

This notebook installs the artifact, runs the functional and adversarial tests, demonstrates the cross-epoch attack of Section 3.2 and why $\Pi_{\mathsf{GR}}$ prevents it, reproduces the exponentiation counts and the exact complexity formula, recomputes every percentage quoted in the paper from the committed benchmark data, redraws Figure 3, runs a short timing benchmark on this machine, and runs an exhaustive checker.

**Runtime.** Sections 1–8 take about 5 minutes on a standard Colab CPU runtime; the exhaustive checker in Section 9 adds about 4 minutes. The full repeated-run benchmark of the paper (Section 10) takes about 1.5 hours and is off by default.

Timings depend on the machine; the paper's timings were measured on an Intel Xeon at 2.10 GHz (see `README.md`). Exponentiation counts are deterministic and must match exactly.

## 0. Get the code

In [ ]:
import os, subprocess, sys
REPO_URL = "https://github.com/<your-username>/proactive-essential-redistribution.git"   # <-- set to your repository
REPO_DIR = "/content/proactive-essential-redistribution" if os.path.isdir("/content") else os.path.abspath("proactive-essential-redistribution")

if not os.path.isdir(REPO_DIR):
    if "<your-username>" in REPO_URL:
        raise SystemExit("Set REPO_URL to your GitHub repository, or upload and unzip the repository so that REPO_DIR exists.")
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
print("working directory:", os.getcwd())
print(sorted(os.listdir(".")))

## 1. Install dependencies

The group operations use **libsodium** (ristretto255) through `ctypes`, and the authority signs epoch records with Ed25519 from **PyNaCl**.

In [ ]:
import shutil, platform
if shutil.which("apt-get"):
    subprocess.run("apt-get -qq update && apt-get -qq install -y libsodium23 > /dev/null", shell=True, check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

import ctypes, nacl
from esr import group
group._lib.sodium_version_string.restype = ctypes.c_char_p
print("Python   ", platform.python_version())
print("libsodium", group._lib.sodium_version_string().decode())
print("PyNaCl   ", nacl.__version__)
print("CPU      ", next((l.split(":",1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name")), platform.processor()))

## 2. Functional and adversarial tests (Experiments 1 and 2)

Correctness over threshold sequences, rejection of unauthorized and stale share sets, malicious dealers, and the three attacks on the insecure shortcuts (Propositions 42–44), which must succeed against the insecure variants and fail against the repaired design.

In [ ]:
r = subprocess.run([sys.executable, "-m", "esr.tests"], capture_output=True, text=True)
print(r.stdout[-4000:]); print(r.stderr[-2000:])
assert "ALL TESTS PASSED" in r.stdout

## 3. The worked example of Section 3.2

Field $\mathbb F_{11}$, custodian $M$, ordinary parties $P_1,\dots,P_4$, threshold $k=3$, key $K=7$. If a window re-shares the ordinary shares but leaves the custodian's share unchanged, an adversary that corrupts only $M$ in one epoch and only $P_3,P_4$ in the next recovers $K$, although neither epoch contains an authorized set.

In [ ]:
p = 11
def interp0(points):                       # Lagrange interpolation at 0 over F_p
    total = 0
    for i, yi in points.items():
        lam = 1
        for j in points:
            if j != i:
                lam = lam * j * pow(j - i, -1, p) % p
        total = (total + lam * yi) % p
    return total

K, m = 7, 4
s  = {i: (3 + 5*i) % p for i in range(1, 5)}     # s(x) = (K - m) + 5x
s2 = {i: (3 + 9*i) % p for i in range(1, 5)}     # re-shared ordinary polynomial, same constant term
print("epoch-0 ordinary shares:", s, " reconstruct with M, P1, P2:", (m + interp0({1: s[1], 2: s[2]})) % p)
print("epoch-1 ordinary shares:", s2)
leak = (m + interp0({3: s2[3], 4: s2[4]})) % p
print("adversary: m from epoch 0 + interpolation of P3, P4 in epoch 1 =", leak, "(key is", K, ")")
assert leak == K

The same attack on real ristretto255 executions: against the shortcut without custodian refresh it recovers the key; against $\Pi_{\mathsf{GR}}$, where every qualified holder contributes fresh randomness to the custodian's new share, it does not.

In [ ]:
from esr.protocol import ESRSystem
from esr.group import rand_scalar, L
from esr.tests import interp0 as interp0_L

def cross_epoch_attack(protocol):
    K = rand_scalar(); S = ESRSystem(4, 1, 1); S.deal(K, 3)
    m_epoch0 = S.phi[1][0]                                  # corrupt M in epoch 0
    if protocol == "shortcut":
        S.window(3, variant="no_M_refresh")
    else:
        S.window_generic(3)                                 # Pi_GR
    sigma = interp0_L({3: S.s[3][0], 4: S.s[4][0]})        # corrupt P3, P4 in epoch 1
    return (m_epoch0 + sigma) % L == K

print("shortcut without custodian refresh: key recovered =", cross_epoch_attack("shortcut"))
print("Pi_GR:                              key recovered =", cross_epoch_attack("pigr"))

## 4. Exponentiation counts are deterministic (Tables 7 and 8)

Re-run the single-run benchmarks for small $n$ in a scratch directory and check that every count equals the committed CSV files.

In [ ]:
import csv, tempfile
tmp = tempfile.mkdtemp()
env = dict(os.environ, PYTHONPATH=REPO_DIR)
for mod in ["esr.bench_gr", "esr.bench_birkhoff", "esr.bench_baseline_dec"]:
    subprocess.run([sys.executable, "-m", mod, "8", "16", "32"], cwd=tmp, env=env, check=True, capture_output=True)

def counts(path):
    return {(r["case"], r["n"]): r["window_exps"] for r in csv.DictReader(open(path))}
ok = True
for fn in ["bench_gr.csv", "bench_birkhoff.csv", "bench_baseline_dec.csv"]:
    new, ref = counts(os.path.join(tmp, fn)), counts(fn)
    for key, v in new.items():
        same = (ref[key] == v); ok &= same
        print(f"{fn:24s} n={key[1]:>3}  {key[0][:55]:55s} {v:>8}  {'match' if same else 'MISMATCH ' + ref[key]}")
assert ok

## 5. The exact complexity formula of Table 6

The per-window exponentiation count of $\Pi_{\mathsf{GR}}$ derived from Algorithm 1 reproduces every measured count exactly.

In [ ]:
def predicted_exps(n, c, tM, k_old, k_new):
    N, tau, de, ds = n - c, tM - 1, k_old - 2, k_new - 2
    dealing = n * (2 * (tau + 1 + ds) + 1)
    verification = sum(n * (2 + (tau if j != 1 else 0)) for j in range(1, c + 1)) \
                 + sum(n * (2 + (ds if i != 1 else 0)) for i in range(1, N + 1))
    nonunit_weights = (tM - (1 if tM == 1 else 0)) + (de + 1 - (1 if de % 2 == 0 else 0))
    authority = (c - 1) * tau + (N - 1) * de + (tau + ds + 2) * nonunit_weights
    return dealing + verification + authority

rows = list(csv.DictReader(open("bench_gr.csv")))
for r in rows:
    c, tM = (3, 2) if "committee" in r["case"] else (1, 1)
    pred = predicted_exps(int(r["n"]), c, tM, int(r["k1"]), int(r["k2"]))
    assert pred == int(r["window_exps"]), (r["case"], r["n"], pred, r["window_exps"])
print(f"formula matches all {len(rows)} measured Pi_GR counts exactly")

## 6. Recompute every percentage quoted in the paper

`summarize_stats.py` recomputes mean, SD, median and 95% confidence intervals from the 266 committed timed runs; `compare.py` recomputes all percentages (written to `bench_logs/compare.txt`).

In [ ]:
subprocess.run([sys.executable, "-m", "esr.summarize_stats"], check=True, capture_output=True)
r = subprocess.run([sys.executable, "esr/compare.py"], capture_output=True, text=True, check=True)
print(r.stdout)

## 7. Figure 3

In [ ]:
subprocess.run([sys.executable, "-m", "esr.plot"], check=True)
from IPython.display import Image, display
display(Image("resharing_cost.png", width=900))

## 8. Short timing benchmark on this machine

Three runs each for $n=8,16,32$ (about 1 minute). Absolute times depend on the machine; compare the relative ordering with Table 7 and Figure 3.

In [ ]:
import importlib, statistics, time
from esr import bench_stats
importlib.reload(bench_stats)
results = []
for n in [8, 16, 32]:
    for label, a, b, fn in bench_stats.cases(n):
        ts = [fn()[0] / n * 1000 for _ in range(3)]
        results.append((n, label, statistics.mean(ts), statistics.stdev(ts)))
for n, label, mean, sd in results:
    print(f"n={n:3d}  {label:62s} {mean:8.2f} ± {sd:5.2f} ms/party")

## 9. Exhaustive checker (Appendix A)

Enumerates every value of the secret and of all coins for one window of $\Pi_{\mathsf{GR}}$ over $\mathbb F_5$ with $N=3$ and checks that no maximal admissible schedule leaks (about 4 minutes). The other checkers run the same way: `verify_resharing.py` (about 7 minutes) and `verify_generic_committee.py` (about 2 minutes).

In [ ]:
RUN_CHECKER = True
if RUN_CHECKER:
    t = time.time()
    r = subprocess.run([sys.executable, "verify_generic_dj.py"], capture_output=True, text=True, check=True)
    print(r.stdout); print(f"{time.time()-t:.0f} s")
    assert r.stdout == open("checker_logs/verify_generic_dj.log").read(), "output differs from the committed log"
    print("output identical to checker_logs/verify_generic_dj.log")

## 10. Optional: full repeated-run benchmark of the paper

10 runs per configuration for $n\le32$, 5 for $n=64$, 3 for $n=128$; about 1.5 hours on a 2-vCPU machine. It writes to a scratch directory so the committed data are not overwritten.

In [ ]:
RUN_FULL_BENCHMARK = False
if RUN_FULL_BENCHMARK:
    out = tempfile.mkdtemp()
    subprocess.run([sys.executable, "-m", "esr.bench_stats", "8", "16", "32", "64", "128"], cwd=out, env=env, check=True)
    for fn in ["bench_gr.csv", "bench_birkhoff.csv", "bench_results.csv", "bench_baseline_dec.csv"]:
        shutil.copy(fn, out)
    os.makedirs(os.path.join(out, "bench_logs"), exist_ok=True)
    subprocess.run([sys.executable, "-m", "esr.summarize_stats"], cwd=out, env=env, check=True)
    print(open(os.path.join(out, "bench_logs", "stats_summary.txt")).read())